In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import cv2
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

#load model and processor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#image preprocessing
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize((0.4815, 0.4578, 0.4082), (0.2686, 0.2613, 0.2758))
])

def vigi_clip_fine_tuning(video_path, text_label, k_percent=0.3, stride=1):
    """
    fine-tunes clip model on one video using attention-based frame selection
    
    args:
        video_path (str): path to video file
        text_label (str): descriptive text (example: "driving is abnormal because the driver is texting")
        k_percent (float): top-k percent of frames to select (default: 0.3)
        stride (int): frame sampling interval (default: 1)
    
    returns:
        torch.Tensor: contrastive loss between video and text
    """

    #step 1: extract frames from video
    cap = cv2.VideoCapture(video_path)
    frames = []
    idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if idx % stride == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            frames.append(pil_image)
        idx += 1
    cap.release()

    if len(frames) == 0:
        raise ValueError("no frames found in video")

    #step 2: get attention scores from vision encoder
    inputs = processor(images=frames, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model.vision_model(pixel_values=inputs["pixel_values"], output_attentions=True)
        attn_scores = outputs.attentions[-1]  # last layer attention

    #step 3: get [cls] token attention to patch tokens
    cls_attn = attn_scores.mean(dim=1)[:, 0, 1:]  # average heads, cls to patch
    frame_scores = cls_attn.mean(dim=1)  # mean across patches

    #step 4: select top-k frames based on attention
    k = max(1, int(k_percent * len(frames)))
    top_indices = torch.topk(frame_scores, k=k).indices
    selected_images = [frames[i] for i in top_indices]

    #step 5: encode selected frames and text
    image_inputs = processor(images=selected_images, return_tensors="pt", padding=True).to(device)
    text_inputs = processor(text=[text_label], return_tensors="pt", padding=True).to(device)

    image_embeds = model.get_image_features(**image_inputs)
    text_embeds = model.get_text_features(**text_inputs)

    #step 6: average frame features (temporal pooling)
    video_embed = image_embeds.mean(dim=0, keepdim=True)

    #step 7: normalize and compute contrastive loss
    video_embed = F.normalize(video_embed, dim=-1)
    text_embed = F.normalize(text_embeds, dim=-1)
    similarity = (video_embed @ text_embed.T) / 0.07  # tau = 0.07
    labels = torch.arange(similarity.shape[0], device=device)
    loss = F.cross_entropy(similarity, labels)

    return loss

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import cv2
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

#load model and processor (must be the fine-tuned model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#image preprocessing
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize((0.4815, 0.4578, 0.4082), (0.2686, 0.2613, 0.2758))
])

def vigi_clip_inference_method1(video_path, candidate_labels, k_percent=0.3, stride=1):
    """
    performs inference using vigi-clip attention-based frame selection
    
    args:
        video_path (str): path to video file
        candidate_labels (list of str): list of descriptive labels to choose from
        k_percent (float): top-k percent of frames to select (default: 0.3)
        stride (int): frame sampling interval (default: 1)
    
    returns:
        str: predicted label with highest similarity
    """

    #step 1: extract frames from video
    cap = cv2.VideoCapture(video_path)
    frames = []
    idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if idx % stride == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            frames.append(pil_image)
        idx += 1
    cap.release()

    if len(frames) == 0:
        raise ValueError("no frames found in video")

    #step 2: get attention scores from vision encoder
    inputs = processor(images=frames, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model.vision_model(pixel_values=inputs["pixel_values"], output_attentions=True)
        attn_scores = outputs.attentions[-1]

    #step 3: get [cls] token attention to patch tokens
    cls_attn = attn_scores.mean(dim=1)[:, 0, 1:]
    frame_scores = cls_attn.mean(dim=1)

    #step 4: select top-k frames
    k = max(1, int(k_percent * len(frames)))
    top_indices = torch.topk(frame_scores, k=k).indices
    selected_images = [frames[i] for i in top_indices]

    #step 5: encode selected frames
    image_inputs = processor(images=selected_images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        image_embeds = model.get_image_features(**image_inputs)
        video_embed = image_embeds.mean(dim=0, keepdim=True)  # temporal pooling
        video_embed = F.normalize(video_embed, dim=-1)

    #step 6: encode all candidate text labels
    text_inputs = processor(text=candidate_labels, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_embeds = model.get_text_features(**text_inputs)
        text_embeds = F.normalize(text_embeds, dim=-1)

    #step 7: compute similarity scores and get best match
    similarity = video_embed @ text_embeds.T  # [1, num_labels]
    best_idx = similarity.argmax().item()
    return candidate_labels[best_idx]

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import cv2
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

#load model and processor (must be the fine-tuned model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

#image preprocessing
resize_dim = 224
transform = T.Compose([
    T.Resize((resize_dim, resize_dim)),
    T.ToTensor(),
    T.Normalize((0.4815, 0.4578, 0.4082), (0.2686, 0.2613, 0.2758))
])

def spatial_crop(image, crop_type="top-left"):
    """returns a cropped image region"""
    w, h = image.size
    crop_w, crop_h = int(w * 0.75), int(h * 0.75)

    if crop_type == "top-left":
        return image.crop((0, 0, crop_w, crop_h))
    elif crop_type == "top-right":
        return image.crop((w - crop_w, 0, w, crop_h))
    elif crop_type == "bottom-left":
        return image.crop((0, h - crop_h, crop_w, h))
    elif crop_type == "bottom-right":
        return image.crop((w - crop_w, h - crop_h, w, h))
    else:
        return image  # fallback: return original

def vigi_clip_inference_method2(video_path, candidate_labels, num_splits=4, frames_per_split=16):
    """
    performs inference using vigi-clip with temporal splitting + spatial cropping
    
    args:
        video_path (str): path to video file
        candidate_labels (list of str): list of descriptive labels to choose from
        num_splits (int): how many time segments to split the video into (default: 4)
        frames_per_split (int): how many frames to sample per split (default: 16)
    
    returns:
        str: predicted label with highest average similarity
    """

    #step 1: extract all frames
    cap = cv2.VideoCapture(video_path)
    all_frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        all_frames.append(pil_image)
    cap.release()

    if len(all_frames) < num_splits * frames_per_split:
        raise ValueError("not enough frames for splitting")

    #step 2: prepare text embeddings
    text_inputs = processor(text=candidate_labels, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        text_embeds = model.get_text_features(**text_inputs)
        text_embeds = F.normalize(text_embeds, dim=-1)

    #step 3: split video into time segments
    segment_size = len(all_frames) // num_splits
    similarity_scores = []

    for s in range(num_splits):
        start = s * segment_size
        end = start + segment_size
        segment_frames = all_frames[start:end]
        
        if len(segment_frames) < frames_per_split:
            continue  # skip if not enough frames

        indices = torch.linspace(0, len(segment_frames)-1, steps=frames_per_split).long()
        sampled_frames = [segment_frames[i] for i in indices]

        #step 4: apply spatial cropping to each frame
        for crop_type in ["top-left", "top-right", "bottom-left", "bottom-right"]:
            cropped_images = [spatial_crop(f, crop_type) for f in sampled_frames]

            #step 5: encode cropped frames
            image_inputs = processor(images=cropped_images, return_tensors="pt", padding=True).to(device)
            with torch.no_grad():
                image_embeds = model.get_image_features(**image_inputs)
                video_embed = image_embeds.mean(dim=0, keepdim=True)  # temporal pooling
                video_embed = F.normalize(video_embed, dim=-1)

                #step 6: compute similarity scores
                sim = (video_embed @ text_embeds.T)  # [1, num_labels]
                similarity_scores.append(sim)

    #step 7: average similarity scores and get best label
    if not similarity_scores:
        raise ValueError("no valid crops or frames to evaluate")

    avg_similarity = torch.mean(torch.cat(similarity_scores, dim=0), dim=0)  # [num_labels]
    best_idx = avg_similarity.argmax().item()
    return candidate_labels[best_idx]

In [ ]:
#video prediction


import os
import math
import time
import cv2
import torch
import numpy as np
from decord import VideoReader, cpu

from utils.config import get_config
from utils.logger import create_logger
from trainers import vificlip




# -- USER SETTINGS --
CONFIG_PATH     = 'from your saved model'
CHECKPOINT_PATH = 'from your saved model' 
INPUT_VIDEO     = r"... # input video
OUTPUT_VIDEO    = r"..." 
SEGMENT_SECONDS = 2
BANNER_POS      = 'top'  # 'top' or 'bottom'
# -----------------------------------

#class names (exactly the same 16 used in training, same order)
CLASS_NAMES = [
    "Driving is normal and the driver is Safely driving",
    "Driving is abnormal because the driver is Doing hair and makeup",
    "Driving is abnormal because the driver is Adjusting radio",
    "Driving is abnormal because the driver is GPS operating",
    "Driving is abnormal because the driver is Writing message using right hand",
    "Driving is abnormal because the driver is Writing message using left hand",
    "Driving is abnormal because the driver is Talking phone using right hand",
    "Driving is abnormal because the driver is Talking phone using left hand",
    "Driving is abnormal because the driver is Having picture",
    "Driving is abnormal because the driver is Talking to passenger",
    "Driving is abnormal because the driver is Singing or dancing",
    "Driving is abnormal because the driver is Fatigue and somnolence",
    "Driving is abnormal because the driver is Drinking using right hand",
    "Driving is abnormal because the driver is Drinking using left hand",
    "Driving is abnormal because the driver is Reaching behind",
    "Driving is abnormal because the driver is Smoking"
]

#preprocess params (ImageNet style; RGB)
IMG_MEAN = np.array([123.675, 116.28, 103.53], dtype=np.float32)
IMG_STD  = np.array([58.395, 57.12, 57.375], dtype=np.float32)

def resize_short_side(img, short_side):
    h, w, _ = img.shape
    if h <= w:
        new_h = short_side
        new_w = int(round(w * short_side / h))
    else:
        new_w = short_side
        new_h = int(round(h * short_side / w))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

def center_crop_to(img, size):
    h, w, _ = img.shape
    th = tw = size
    y1 = max((h - th) // 2, 0)
    x1 = max((w - tw) // 2, 0)
    return img[y1:y1+th, x1:x1+tw, :]

def preprocess_clip_rgb(frames_rgb, input_size):
    """
    frames_rgb: list of RGB frames (uint8), HxWx3
    Output tensor: (1, T, C, H, W)
    """
    scale_resize = int(256 / 224 * input_size)
    proc = []
    for img in frames_rgb:
        img = resize_short_side(img, scale_resize)
        img = center_crop_to(img, input_size)
        img = img.astype(np.float32)
        img = (img - IMG_MEAN) / IMG_STD
        proc.append(img)
    arr = np.stack(proc, axis=0)           # (T, H, W, C)
    arr = np.transpose(arr, (0, 3, 1, 2))  # (T, C, H, W)
    arr = np.expand_dims(arr, axis=0)      # (1, T, C, H, W)
    return torch.from_numpy(arr).float()

def uniform_indices(start_idx, end_idx, num):
    if end_idx < start_idx:
        end_idx = start_idx
    if num <= 1:
        return [start_idx]
    lin = np.linspace(start_idx, end_idx, num=num)
    inds = np.clip(np.round(lin).astype(int), start_idx, end_idx).tolist()
    return inds

def draw_label_banner(frame, text, pos='top', max_w_ratio=0.95):
    """
    Draw a translucent banner and wrap text into multiple lines so it's fully visible.
    """
    h, w = frame.shape[:2]
    margin = 12
    max_w = int(w * max_w_ratio)

    base_font_scale = max(w / 1280.0, 0.6)
    font = cv2.FONT_HERSHEY_SIMPLEX
    thickness = 2

    # word wrap
    words = text.split()
    lines, cur = [], ""
    for word in words:
        test = word if not cur else f"{cur} {word}"
        (tw, th), _ = cv2.getTextSize(test, font, base_font_scale, thickness)
        if tw + 2 * margin <= max_w:
            cur = test
        else:
            if cur:
                lines.append(cur)
            cur = word
    if cur:
        lines.append(cur)

    # banner height
    (_, th), baseline = cv2.getTextSize("Ag", font, base_font_scale, thickness)
    line_h = th + baseline + 4
    banner_h = line_h * len(lines) + margin * 2

    # position
    y0 = 0 if pos == 'top' else h - banner_h
    x0 = 0

    # translucent banner
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (w, y0 + banner_h), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.45, frame, 0.55, 0.0)

    # draw text
    y = y0 + margin + th
    for line in lines:
        cv2.putText(frame, line, (margin, y), font, base_font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
        y += line_h
    return frame

def main():
    #build config + logger
    class Args:
        def __init__(self):
            self.config = CONFIG_PATH
            self.output = "output_vificlip"
            self.resume = None
            self.only_test = True
            self.opts = None
            self.batch_size = 1
            self.pretrained = None
            self.accumulation_steps = 1
            self.local_rank = 0

    args = Args()
    config = get_config(args)
    os.makedirs(args.output, exist_ok=True)
    logger = create_logger(output_dir=args.output, name=f"{config.MODEL.ARCH}")
    logger.info(f"Inference on: {INPUT_VIDEO}")
    
    #load model
    model = vificlip.returnCLIP(config, logger=logger, class_names=CLASS_NAMES).float().cuda().eval()
    state = torch.load(CHECKPOINT_PATH, map_location='cuda')
    model.load_state_dict(state, strict=False)
    model.eval()

    #open input via Decord
    vr = VideoReader(INPUT_VIDEO, ctx=cpu(0))
    num_frames = len(vr)
    fps = vr.get_avg_fps()
    if fps is None or fps <= 0:
        cap = cv2.VideoCapture(INPUT_VIDEO)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        cap.release()
    fps = float(fps)
    seg_len_frames = int(round(SEGMENT_SECONDS * fps))

    #output size via OpenCV
    cap0 = cv2.VideoCapture(INPUT_VIDEO)
    width  = int(cap0.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap0.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap0.release()

    #writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

    #preload frames in both RGB (for model) and BGR (for writing)
    all_frames_rgb = []
    all_frames_bgr = []
    for i in range(num_frames):
        rgb = vr[i].asnumpy()                  # RGB uint8
        bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        all_frames_rgb.append(rgb)
        all_frames_bgr.append(bgr)

    T_model = int(config.DATA.NUM_FRAMES)
    input_size = int(config.DATA.INPUT_SIZE)
    num_segments = math.ceil(num_frames / seg_len_frames)

    with torch.no_grad(), torch.cuda.amp.autocast():
        for seg_idx in range(num_segments):
            seg_start = seg_idx * seg_len_frames
            seg_end   = min((seg_idx + 1) * seg_len_frames - 1, num_frames - 1)

            #sample frames uniformly for this segment
            sample_inds = uniform_indices(seg_start, seg_end, T_model)
            sampled_rgb = [all_frames_rgb[i] for i in sample_inds]

            #preprocess to (1, T, C, H, W)
            clip_tensor = preprocess_clip_rgb(sampled_rgb, input_size).cuda(non_blocking=True)

            # --model-only inference time (ms) --
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            logits = model(clip_tensor)                     # (1, num_classes)

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            infer_ms = (t1 - t0) * 1000.0
            # ------------------------------------------------

            probs  = torch.softmax(logits, dim=-1)[0].float()
            pred_idx = int(torch.argmax(probs).item())
            conf = float(probs[pred_idx].item())

            label_text = f"{CLASS_NAMES[pred_idx]}  ({conf*100:.1f}%)   infer: {infer_ms:.1f} ms"

            #overlay label on all frames in this segment (same behavior as before)
            for fidx in range(seg_start, seg_end + 1):
                frame = all_frames_bgr[fidx].copy()
                frame = draw_label_banner(frame, label_text, pos=BANNER_POS, max_w_ratio=0.95)
                out.write(frame)

    out.release()
    print(f"Saved labeled video to: {os.path.abspath(OUTPUT_VIDEO)}")

if __name__ == "__main__":
    main()
